## Group No 44

## Group Member Names:
1. Kaustav  Dey (2024aa05530@wilp.bits-pilani.ac.in) % contribution 100% 
2. NEERAJ KUMAR TALREJA (2024ab05141@wilp.bits-pilani.ac.in) % contribution 100%
3. MOHAMED NASIR MH (2024AB05052@wilp.bits-pilani.ac.in) % contribution 100%
4. Mithun M Pillai (2024AB05217@wilp.bits-pilani.ac.in) % contribution 100%
5. MD. SHAFI HUSSAIN (2024ab05039@wilp.bits-pilani.ac.in) % contribution 0%

In [1]:
# =========================
# Scaling Configuration
# =========================

TARGET_TOTAL_QUESTIONS = 100

TARGET_PER_TYPE_PER_RUN = 2   # safe batch size

MAX_CONTEXT_CHARS = 1200

GENERATION_OUTPUT_FILE = "qa_eval_dataset.json"

EVAL_SUBSET_SIZE = 20   # safe evaluation size

TOP_K_RETRIEVAL = 2


## Install Required Packages

In [2]:

!pip install llama-index-core
!pip install llama-index-embeddings-huggingface
!pip install llama-index-retrievers-bm25
!pip install llama-index-llms-groq
!pip install beautifulsoup4 requests transformers


  Using cached llama_index_llms_groq-0.4.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached llama_index_llms_openai_like-0.5.3-py3-none-any.whl.metadata (1.1 kB)
  Using cached llama_index_llms_openai-0.6.18-py3-none-any.whl.metadata (3.0 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.7
    Uninstalling huggingface_hub-1.3.7:
      Successfully uninstalled huggingface_hub-1.3.7
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Imports

In [3]:

import os
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, unquote
from transformers import AutoTokenizer

from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq


## Global Configuration (Embedding +  LLM)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# Set your Groq API key before running generation


FIXED_URL_FILE = "/content/drive/MyDrive/Assignment2/fixed_urls.json"
FIXED_COUNT = 200
RANDOM_COUNT = 300
MIN_WORDS = 200
WIKI_RANDOM_URL = "https://en.wikipedia.org/wiki/Special:Random"

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)
#changing from groq to tiny
#Settings.llm = Groq(
#    model="llama-3.1-8b-instant",
#    temperature=0.0
#)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# =========================
# TinyLlama Local LLM Setup 
# =========================

!pip install -q transformers accelerate bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading TinyLlama model...")

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

generator = pipeline(
    "text-generation",
    model=model_llm,
    tokenizer=tokenizer_llm,
    max_new_tokens=48,
    do_sample=False,
    truncation=True,
    pad_token_id=tokenizer_llm.eos_token_id
)

print("TinyLlama loaded successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.6 MB/s eta 0:00:00
Loading TinyLlama model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TinyLlama loaded successfully.


In [7]:
from llama_index.core.llms import CustomLLM, CompletionResponse, LLMMetadata
from llama_index.core import Settings


class TinyLlamaLLM(CustomLLM):

    @property
    def metadata(self) -> LLMMetadata:
        return LLMMetadata(
            context_window=2048,
            num_output=128,
            model_name="TinyLlama-1.1B-Chat"
        )

    def complete(self, prompt: str, **kwargs) -> CompletionResponse:

        result = generator(prompt)

        text = result[0]["generated_text"]

        if text.startswith(prompt):
            text = text[len(prompt):]

        return CompletionResponse(text=text.strip())

    def stream_complete(self, prompt: str, **kwargs):
        # Minimal implementation required by LlamaIndex
        yield self.complete(prompt)


# Set TinyLlama as active LLM
Settings.llm = TinyLlamaLLM()

print("TinyLlama successfully configured as LlamaIndex LLM")



TinyLlama successfully configured as LlamaIndex LLM


## Helper Functions

In [8]:

def normalize_wiki_url(url):
    parsed = urlparse(url)
    return f"https://en.wikipedia.org{unquote(parsed.path)}"


In [9]:

def fetch_wikipedia_page(url):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        if r.status_code != 200:
            return None, None
        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.find("h1").text.strip()
        text = " ".join(p.get_text() for p in soup.select("div.mw-parser-output > p"))
        if len(text.split()) < MIN_WORDS:
            return None, None
        return title, normalize_wiki_url(url)
    except:
        return None, None


## URL Collection

In [10]:

def create_or_load_fixed_urls():

    print(f"Target fixed URLs: {FIXED_COUNT}")

    if os.path.exists(FIXED_URL_FILE):

        print("Loading fixed URLs from file")

        with open(FIXED_URL_FILE) as f:
            return json.load(f)

    print("Generating fixed URLs...")

    fixed_urls = set()

    while len(fixed_urls) < FIXED_COUNT:

        try:

            r = requests.get(
            WIKI_RANDOM_URL,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=5
             )

            print("Fetched:", r.url)

            _, url = fetch_wikipedia_page(r.url)

            if url:
                fixed_urls.add(url)
                print(f"Fixed URLs: {len(fixed_urls)}/{FIXED_COUNT}")

        except requests.exceptions.RequestException:
            print("Retrying...")



    with open(FIXED_URL_FILE, "w") as f:
        json.dump(list(fixed_urls), f, indent=2)

    print("Saved fixed URLs file.")

    return list(fixed_urls)




In [11]:

def sample_random_urls(fixed_urls):
    random_urls = set()
    fixed = set(fixed_urls)
    while len(random_urls) < RANDOM_COUNT:
        r = requests.get(WIKI_RANDOM_URL, headers={"User-Agent": "Mozilla/5.0"})
        _, url = fetch_wikipedia_page(r.url)
        if url and url not in fixed:
            random_urls.add(url)
            print(f"Random URLs: {len(random_urls)}/{RANDOM_COUNT}")
    return list(random_urls)


## Corpus Construction

In [12]:

fixed_urls = create_or_load_fixed_urls()
random_urls = sample_random_urls(fixed_urls)
corpus_urls = fixed_urls + random_urls
print("Total URLs:", len(corpus_urls))


Target fixed URLs: 200
Loading fixed URLs from file
Random URLs: 1/300
Random URLs: 2/300
Random URLs: 3/300
Random URLs: 4/300
Random URLs: 5/300
Random URLs: 6/300
Random URLs: 7/300
Random URLs: 8/300
Random URLs: 9/300
Random URLs: 10/300
Random URLs: 11/300
Random URLs: 12/300
Random URLs: 13/300
Random URLs: 14/300
Random URLs: 15/300
Random URLs: 16/300
Random URLs: 17/300
Random URLs: 18/300
Random URLs: 19/300
Random URLs: 20/300
Random URLs: 21/300
Random URLs: 22/300
Random URLs: 23/300
Random URLs: 24/300
Random URLs: 25/300
Random URLs: 26/300
Random URLs: 27/300
Random URLs: 28/300
Random URLs: 29/300
Random URLs: 30/300
Random URLs: 31/300
Random URLs: 32/300
Random URLs: 33/300
Random URLs: 34/300
Random URLs: 35/300
Random URLs: 36/300
Random URLs: 37/300
Random URLs: 38/300
Random URLs: 39/300
Random URLs: 40/300
Random URLs: 41/300
Random URLs: 42/300
Random URLs: 43/300
Random URLs: 44/300
Random URLs: 45/300
Random URLs: 46/300
Random URLs: 47/300
Random URLs: 48/3

## Text Cleaning

In [13]:

def extract_clean_wikipedia_text(url):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.find("h1").text.strip()
    content = soup.find("div", class_="mw-parser-output")
    for tag in content.find_all(["table", "sup", "span"]):
        tag.decompose()
    text = " ".join(p.get_text() for p in content.find_all("p"))
    return title, " ".join(text.split())


In [14]:

documents = []
for url in corpus_urls:
    title, text = extract_clean_wikipedia_text(url)
    documents.append({"url": url, "title": title, "text": text})
print("Documents:", len(documents))


Documents: 500


## Token Chunking

In [15]:

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def token_chunk_text(text, min_tokens=200, max_tokens=400, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        if end - start >= min_tokens:
            chunks.append(tokenizer.decode(tokens[start:end]))
        next_start = end - overlap
        if next_start <= start:
            break
        start = next_start
    return chunks


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Nodes

In [16]:

chunked_docs = []
for doc in documents:
    for i, ch in enumerate(token_chunk_text(doc["text"])):
        chunked_docs.append({
            "chunk_id": f"{doc['url']}_chunk_{i}",
            "url": doc["url"],
            "title": doc["title"],
            "text": ch
        })
print("Chunks:", len(chunked_docs))


Token indices sequence length is longer than the specified maximum sequence length for this model (12181 > 512). Running this sequence through the model will result in indexing errors


Chunks: 1506


In [17]:

nodes = []
for c in chunked_docs:
    nodes.append(TextNode(
        text=c["text"],
        metadata={
            "chunk_id": c["chunk_id"],
            "url": c["url"],
            "title": c["title"],
            "source": url
        }
    ))
print("Nodes:", len(nodes))


Nodes: 1506


## Hybrid Retrieval

In [18]:

index = VectorStoreIndex(nodes)
bm25 = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)
dense = index.as_retriever(similarity_top_k=5)

fusion_retriever = QueryFusionRetriever(
    [dense, bm25],
    similarity_top_k=5,
    mode="reciprocal_rerank",
    num_queries=1
)


DEBUG:bm25s:Building index from IDs objects


## Answer Generation

In [19]:

from llama_index.core.prompts import PromptTemplate

#MAX_CONTEXT_CHARS = 1500  # safe for TinyLlama

qa_prompt = PromptTemplate(
    """You are an expert assistant.
Use the following context to answer the question.

Context:
{context}

Question:
{question}

Answer concisely and accurately."""
)

def generate_answer(query, retriever, top_k=2):

    retrieved_nodes = retriever.retrieve(query)[:top_k]

    MAX_CONTEXT_CHARS = 1500

    context = ""
    for i, n in enumerate(retrieved_nodes):

        text = n.node.text

        if len(context) + len(text) > MAX_CONTEXT_CHARS:
            break

        context += f"[{i+1}] {text}\n\n"

    prompt = qa_prompt.format(
        context=context,
        question=query
    )

    response = Settings.llm.complete(prompt)

    return response.text, retrieved_nodes



## Storing DB and Corpus

In [20]:
index.storage_context.persist(
    persist_dir="/content/drive/MyDrive/Assignment2/vector_index"
)


In [21]:
with open("/content/drive/MyDrive/Assignment2/chunked_corpus.json", "w") as f:
    json.dump(chunked_docs, f, indent=2)


## End-to-End Test

In [22]:

query = "What is the role of mitochondria in a cell?"
answer, retrieved = generate_answer(query, fusion_retriever)

print("Answer:\n", answer)
print("\nSources:")
for r in retrieved:
    print("-", r.node.metadata["title"], "|", r.node.metadata["url"])


Answer:
 Question:
What is the function of the Golgi apparatus in a cell?

Answer concisely and accurately.



Question:
What is the function of the endoplasmic

Sources:
- Segment polarity gene | https://en.wikipedia.org/wiki/Segment_polarity_gene
- DDR1 | https://en.wikipedia.org/wiki/DDR1


In [23]:
!pip install gradio


## UI App creation

In [24]:
import gradio as gr
import time

def rag_ui(query):
    start = time.time()

    retrieved_nodes = fusion_retriever.retrieve(query)

    context = "\n\n".join(
        f"[{i+1}] {n.node.text}"
        for i, n in enumerate(retrieved_nodes[:5])
    )

    prompt = f"""
You are an expert assistant.
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer concisely and accurately.
"""

    response = Settings.llm.complete(prompt)
    latency = time.time() - start

    sources = "\n".join(
        f"- {r.node.metadata['title']} | {r.node.metadata['url']}"
        for r in retrieved_nodes[:5]
    )

    return response.text, sources, f"{latency:.2f} seconds"

demo = gr.Interface(
    fn=rag_ui,
    inputs=gr.Textbox(label="Enter your query"),
    outputs=[
        gr.Textbox(label="Generated Answer"),
        gr.Textbox(label="Sources"),
        gr.Textbox(label="Response Time")
    ],
    title="Hybrid RAG System (BM25 + Dense + RRF + Groq)",
    description="Wikipedia-based Hybrid Retrieval-Augmented Generation system"
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://399ef962326b7085cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Answer generation

In [25]:
def generate_answer_from_context(query, context):
    """
    Used ONLY in Part 2 for faithfulness evaluation
    """
    prompt = qa_prompt.format(
        context=context,
        question=query
    )
    response = Settings.llm.complete(prompt)
    return response.text


In [26]:
# =====================
# Extract chunks from Part 1 index (for Part 2)
# =====================

all_nodes = list(index.docstore.docs.values())

all_chunks = [
    {
        "chunk_id": node.node_id,
        "text": node.text,
        "url": node.metadata.get("url")
    }
    for node in all_nodes
]

print(f"Extracted {len(all_chunks)} chunks from index")
print(all_chunks[0])


Extracted 1506 chunks from index
{'chunk_id': '34d6e4c9-da97-4ee3-87b5-e33dd565da1f', 'text': 'filodrillia tricarinata is a species of sea snail, a marine gastropod mollusk in the family borsoniidae. ( original description ) the length of the shell attains 6 mm, its width 2 mm. the small shell has an elongate fusiform shape and contains six convex whorls, including the 1½ whorls of the obtuse nucleus. this is a banded shell without plaits, which is unusual in the genus, but it has three delicate keels, one of which is on the suture, and the spaces between are shallow concave grooves. the aperture is ovate. the sinus is entirely posterior, is wide and deep, and the outer lip is thickened. ( description by verco, 1909 ) this shell varies much in shape and sculpture. there may be three sharp spirals on each of the four spire - whorls in a shell 8 mm. long, or two on the first and second spire - whorls, an intercalated third thread on the third whorl, and three on the fourth whorl. the she

## Automated Question Generation

In [27]:
#from llama_index.core import Settings

#def llm(prompt: str) -> str:
    #"""
    #Adapter to reuse the Part 1 Groq LLM for Part 2
   # """
#    response = Settings.llm.complete(prompt)
#    return response.text


In [28]:
# =========================
# TinyLlama wrapper function
# =========================

def llm(prompt: str) -> str:

    result = generator(prompt)

    text = result[0]["generated_text"]

    # Remove prompt from output if duplicated
    if text.startswith(prompt):
        text = text[len(prompt):]

    return text.strip()


In [29]:
import json
import random
from tqdm import tqdm

FIXED_URL_FILE = "/content/drive/MyDrive/Assignment2/fixed_urls.json"

with open(FIXED_URL_FILE, "r") as f:
    fixed_urls = json.load(f)

fixed_urls = set(fixed_urls)

print(f"Loaded {len(fixed_urls)} fixed Wikipedia URLs")


Loaded 200 fixed Wikipedia URLs


In [30]:
# Assumption: `all_chunks` already exists from Part 1
# all_chunks: List[dict] with keys -> chunk_id, text, url, title

fixed_chunks = [c for c in all_chunks if c["url"] in fixed_urls]

print(f"Total chunks from fixed URLs: {len(fixed_chunks)}")


Total chunks from fixed URLs: 605


## Question Prompts

In [31]:
QUESTION_PROMPTS = {
    "factual": (
        "Generate one factual question whose answer is explicitly stated "
        "in the text. Also provide the exact answer."
    ),
    "comparative": (
        "Generate one comparative question that compares two concepts "
        "mentioned or implied in the text. Provide the answer."
    ),
    "inferential": (
        "Generate one inferential question that requires reasoning "
        "based on the text (not a direct quote). Provide the answer."
    ),
}


### Generate QA from Chunk

In [34]:
def generate_qa_from_chunk(llm, chunk_text, qtype):

    MAX_CONTEXT_CHARS = 1200
    chunk_text = chunk_text[:MAX_CONTEXT_CHARS]

    prompt = f"""
Generate ONE {qtype} question and answer from the text below.

Format exactly as:

Question: <question>
Answer: <answer>

Text:
{chunk_text}
"""

    try:
        response = llm(prompt)

        print("\n=== TinyLlama raw response ===\n", response)

        if not response or len(response.strip()) < 10:
            return None, None

        question = None
        answer = None

        lines = response.split("\n")

        for line in lines:

            line = line.strip()

            if line.lower().startswith("question:"):
                question = line.split(":", 1)[1].strip()

            elif line.lower().startswith("answer:"):
                answer = line.split(":", 1)[1].strip()

        # fallback parsing
        if not question:
            for line in lines:
                if "?" in line:
                    question = line.strip()
                    break

        if not answer and question:
            answer = response.replace(question, "").strip()[:300]

        if not question or not answer:
            return None, None

        return question, answer

    except Exception as e:
        print("Error:", e)
        return None, None



## Creation of Question Answer

In [35]:
# =========================
# Safe scalable Q&A generation controller
# =========================

import os
import json
import random
from tqdm import tqdm
from collections import Counter


GENERATION_OUTPUT_FILE = "/content/drive/MyDrive/Assignment2/qa_eval_dataset.json"

TARGET_TOTAL_QUESTIONS = 60

TARGET_PER_TYPE_PER_RUN = 2


# Load existing dataset if exists
if os.path.exists(GENERATION_OUTPUT_FILE):

    with open(GENERATION_OUTPUT_FILE, "r") as f:
        qa_dataset = json.load(f)

    print(f"Loaded existing dataset with {len(qa_dataset)} questions")

else:

    qa_dataset = []

    print("Starting new dataset")


counts = Counter(q["question_type"] for q in qa_dataset)

question_id = len(qa_dataset)

random.shuffle(fixed_chunks)

GENERATION_CHUNK_LIMIT = 150
generation_chunks = fixed_chunks[:GENERATION_CHUNK_LIMIT]



for chunk in tqdm(generation_chunks):

    if len(qa_dataset) >= TARGET_TOTAL_QUESTIONS:
        break

    qtype = list(QUESTION_PROMPTS.keys())[len(qa_dataset) % len(QUESTION_PROMPTS)]

    if len(qa_dataset) >= TARGET_TOTAL_QUESTIONS:
            break

    question, answer = generate_qa_from_chunk(
            llm,
            chunk["text"],
            qtype
        )

    if question is None:
            continue

    qa_dataset.append({

            "question_id": f"q_{question_id:03d}",
            "question": question,
            "answer": answer,
            "question_type": qtype,
            "source_urls": [chunk["url"]]

        })

    counts[qtype] += 1
    question_id += 1

    print(f"Generated {len(qa_dataset)} / {TARGET_TOTAL_QUESTIONS}")

        # Save progress safely
    with open(GENERATION_OUTPUT_FILE, "w") as f:
            json.dump(qa_dataset, f, indent=2)


print("Generation finished.")
print("Total questions:", len(qa_dataset))


Loaded existing dataset with 37 questions


  1%|          | 1/150 [00:03<07:39,  3.08s/it]


=== TinyLlama raw response ===
 Based on the text material, generate the comparative question and answer:

Question: What is the relationship between state nationalism and popular nationalism in contemporary Chinese nationalism, and how has it evolved over time?
Generated 38 / 60


  1%|▏         | 2/150 [00:04<05:41,  2.31s/it]


=== TinyLlama raw response ===
 Based on the text material above, generate the response to the following quesion or instruction: How did the 1980 New Hampshire primary and the subsequent Super Tuesday primaries contribute to the nomination of


  2%|▏         | 3/150 [00:06<04:30,  1.84s/it]

... (205 outputs left out)

 73%|███████▎  | 109/150 [02:42<00:52,  1.28s/it]


=== TinyLlama raw response ===
 development possible but also allowed the team to create a game that they were proud of.


 73%|███████▎  | 110/150 [02:44<00:59,  1.49s/it]


=== TinyLlama raw response ===
 Generate ONE inferential question and answer from the text below.

Format exactly as:

Question: What was the focus of the magazine's early years, and how did it change over time?
Answer
Generated 60 / 60
Generation finished.
Total questions: 60


In [36]:
MULTI_HOP_TARGET = 0

url_to_chunks = {}
for c in fixed_chunks:
    url_to_chunks.setdefault(c["url"], []).append(c)

urls = list(url_to_chunks.keys())

while len([q for q in qa_dataset if q["question_type"] == "multi-hop"]) < MULTI_HOP_TARGET:
    url1, url2 = random.sample(urls, 2)

    c1 = random.choice(url_to_chunks[url1])
    c2 = random.choice(url_to_chunks[url2])

    prompt = f"""
You are given two Wikipedia passages.

PASSAGE 1:
{c1["text"]}

PASSAGE 2:
{c2["text"]}

TASK:
Generate one multi-hop question that requires information from BOTH passages.
Provide the answer.

Return JSON with:
- question
- answer
"""

    response = llm(prompt)

    try:
        qa = json.loads(response)
    except Exception:
        continue

    qa_dataset.append({
        "question_id": f"q_{question_id:03d}",
        "question": qa["question"],
        "answer": qa["answer"],
        "question_type": "multi-hop",
        "source_urls": [url1, url2]
    })

    question_id += 1


In [37]:
print(f"Total Q&A pairs generated: {len(qa_dataset)}")

from collections import Counter
print("Question type distribution:")
print(Counter([q["question_type"] for q in qa_dataset]))

OUTPUT_PATH = "/content/drive/MyDrive/Assignment2/qa_eval_dataset.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(qa_dataset, f, indent=2)

print(f"Saved evaluation dataset to {OUTPUT_PATH}")


Total Q&A pairs generated: 60
Question type distribution:
Counter({'factual': 20, 'comparative': 20, 'inferential': 20})
Saved evaluation dataset to /content/drive/MyDrive/Assignment2/qa_eval_dataset.json


We created 60 questions instead of 100 as we were facing dificulties in running the question generations scripts in our laptop, it was taking more than 8-9 hours

In [38]:
# =========================
# Safe evaluation runner
# =========================

EVAL_SUBSET_SIZE = 20

subset = qa_dataset[:EVAL_SUBSET_SIZE]

print("Evaluating on", len(subset), "questions")

#results = run_complete_pipeline(subset)


Evaluating on 20 questions


### Automated Evaluation

In [39]:
assert hasattr(fusion_retriever, "retrieve"), "Retriever corrupted!"


In [40]:
def hybrid_retrieve(query: str, top_k: int = 2):
    """
    Adapter for LlamaIndex QueryFusionRetriever
    Returns ranked chunks as dicts with text and url
    """
    nodes = fusion_retriever.retrieve(query)

    results = []
    for n in nodes[:top_k]:
        node = n.node
        results.append({
            "text": node.text,
            "url": node.metadata.get("url"),
            "score": n.score
        })

    return results


In [41]:
import json
import numpy as np
from collections import defaultdict
from tqdm import tqdm

EVAL_DATASET_PATH = "/content/drive/MyDrive/Assignment2/qa_eval_dataset.json"

with open(EVAL_DATASET_PATH, "r") as f:
    eval_dataset = json.load(f)

print(f"Loaded {len(eval_dataset)} evaluation questions")


Loaded 60 evaluation questions


In [42]:
import pandas as pd

df = pd.DataFrame(qa_dataset)

df.to_csv(
    "/content/drive/MyDrive/Assignment2/qa_eval_dataset.csv",
    index=False
)


In [43]:
def retrieve_ranked_urls(query, top_k=5):
    """
    Runs hybrid retrieval and converts ranked chunks → ranked URLs
    """
    retrieved_chunks = hybrid_retrieve(query, top_k=top_k)

    ranked_urls = []
    seen = set()

    for chunk in retrieved_chunks:
        url = chunk["url"]
        if url not in seen:
            ranked_urls.append(url)
            seen.add(url)

    return ranked_urls


## Compute MRR

In [44]:
def compute_url_mrr(eval_dataset, top_k=10):
    reciprocal_ranks = []

    for item in tqdm(eval_dataset):
        query = item["question"]
        gold_urls = set(item["source_urls"])

        ranked_urls = retrieve_ranked_urls(query, top_k=top_k)

        rank = None
        for idx, url in enumerate(ranked_urls, start=1):
            if url in gold_urls:
                rank = idx
                break

        if rank is not None:
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    return float(np.mean(reciprocal_ranks))


In [45]:
mrr_score = compute_url_mrr(eval_dataset)
print(f"URL-level MRR: {mrr_score:.4f}")


100%|██████████| 60/60 [00:03<00:00, 15.78it/s]

URL-level MRR: 0.9000


Justification for Selection

Recall@K is a fundamental metric for evaluating the retrieval effectiveness of a Retrieval-Augmented Generation (RAG) system. In a RAG pipeline, the generator can only produce correct and grounded answers if the retriever successfully fetches the relevant source documents. Therefore, evaluating retrieval recall is essential to determine whether the retriever provides sufficient supporting evidence.

Recall@K is particularly important in our system because:

Our RAG pipeline uses hybrid retrieval (Dense + BM25 + Reciprocal Rank Fusion), and Recall@K measures whether this fusion successfully retrieves relevant Wikipedia sources.

Many questions, especially multi-hop and inferential questions, require multiple relevant documents. Recall@K measures whether all required sources are retrieved.

Unlike MRR, which considers only the first correct result, Recall@K evaluates overall retrieval coverage, making it more suitable for assessing complex reasoning tasks.

Mathematical Formulation

Let:

Q = total number of evaluation questions

G_i = set of ground-truth source URLs for question i

R_i@K = set of top-K retrieved URLs for question i

The Recall@K for a single question i is defined as:

Recall@K_i = | G_i ∩ R_i@K | / | G_i |

where:

| G_i ∩ R_i@K | = number of correctly retrieved URLs within top K

| G_i | = total number of relevant URLs for that question

The overall Recall@K across all questions is computed as the average:

Recall@K = (1 / Q) × SUM over i=1 to Q of Recall@K_i

Higher Recall@K indicates that the retriever successfully provides relevant evidence, increasing the likelihood of correct and grounded answer generation.

Low Recall@K suggests:

Relevant documents are missing

Retriever embeddings may be insufficient

Fusion strategy may need tuning

### Compute Recall

In [46]:
def compute_recall_at_k(eval_dataset, k=10):
    recalls = []

    for item in tqdm(eval_dataset):
        query = item["question"]
        gold_urls = set(item["source_urls"])

        ranked_urls = retrieve_ranked_urls(query, top_k=k)
        retrieved_set = set(ranked_urls[:k])

        recall = len(retrieved_set & gold_urls) / len(gold_urls)
        recalls.append(recall)

    return float(np.mean(recalls))


In [47]:
recall_at_10 = compute_recall_at_k(eval_dataset, k=10)
print(f"Recall@10 (URL-level): {recall_at_10:.4f}")


100%|██████████| 60/60 [00:04<00:00, 13.79it/s]

Recall@10 (URL-level): 0.9000


Justification for Selection

Faithfulness measures whether the generated answer is fully supported by the retrieved context, which is critical for evaluating hallucination in RAG systems.

Unlike traditional generation metrics such as BLEU or ROUGE, which measure textual similarity, faithfulness evaluates whether the answer is factually grounded in retrieved documents.

Faithfulness is particularly important because:

The primary purpose of RAG systems is to reduce hallucination.

Even if retrieval is correct, the generator may produce unsupported or fabricated information.

Faithfulness ensures that answers are derived only from retrieved Wikipedia sources.

This metric directly evaluates the core goal of RAG: grounded and reliable generation.

Let:

Q = total number of evaluation questions

f_i = faithfulness score for question i

The faithfulness score for each question is defined as:

f_i = 1, if the generated answer is fully supported by the retrieved context
f_i = 0, if the answer contains hallucinated or unsupported information

The overall Faithfulness score is computed as:

Faithfulness = (1 / Q) × SUM over i=1 to Q of f_i

High faithfulness indicates:

Effective retrieval

Proper grounding

Reliable answer generation

Low faithfulness indicates:

Hallucination

Poor retrieval coverage

Context misunderstanding

## Compute Faithfulness

In [48]:
def faithfulness_check(llm, question, answer, contexts):
    prompt = f"""
You are a fact-checker.

QUESTION:
{question}

ANSWER:
{answer}

CONTEXT:
{contexts}

Is the answer fully supported by the context?
Respond with only one word: YES or NO.
"""
    response = llm(prompt).strip().lower()
    return 1 if "yes" in response else 0


In [49]:
def compute_faithfulness(eval_dataset, top_k=2):
    scores = []

    for item in tqdm(eval_dataset):
        query = item["question"]

        retrieved_chunks = hybrid_retrieve(query, top_k=top_k)
        #contexts = "\n".join([c["text"] for c in retrieved_chunks])

        MAX_CONTEXT_CHARS = 1500

        contexts = ""
        for c in retrieved_chunks:
            if len(contexts) + len(c["text"]) > MAX_CONTEXT_CHARS:
                break
            contexts += c["text"] + "\n"


        generated_answer = generate_answer_from_context(query, contexts)

        score = faithfulness_check(
            llm,
            query,
            generated_answer,
            contexts
        )
        scores.append(score)

    return float(np.mean(scores))


In [50]:
faithfulness_score = compute_faithfulness(eval_dataset[:10])
print(f"Answer Faithfulness Score: {faithfulness_score:.4f}")


100%|██████████| 10/10 [00:28<00:00,  2.86s/it]

Answer Faithfulness Score: 0.6000


In [51]:
results = {
    "URL_MRR": mrr_score,
    "Recall@10": recall_at_10,
    "Faithfulness": faithfulness_score
}

print("=== Final Evaluation Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


=== Final Evaluation Results ===
URL_MRR: 0.9000
Recall@10: 0.9000
Faithfulness: 0.6000


## Innovative Evaluation

### Ablation Study (Dense vs Sparse vs Hybrid)

In [52]:
def retrieve_dense_only(query, top_k=5):
    nodes = dense.retrieve(query)
    return [{"text": n.node.text, "url": n.node.metadata.get("url")} for n in nodes[:top_k]]

def retrieve_sparse_only(query, top_k=5):
    nodes = bm25.retrieve(query)
    return [{"text": n.node.text, "url": n.node.metadata.get("url")} for n in nodes[:top_k]]

def retrieve_hybrid(query, top_k=2):
    return hybrid_retrieve(query, top_k)


In [53]:
def evaluate_retrievers(eval_dataset):

    methods = {
        "dense": retrieve_dense_only,
        "sparse": retrieve_sparse_only,
        "hybrid": retrieve_hybrid
    }

    results = {}

    for name, retriever in methods.items():
        recalls = []

        for item in eval_dataset:
            gold_urls = set(item["source_urls"])
            retrieved = retriever(item["question"], top_k=2)
            retrieved_urls = set(r["url"] for r in retrieved)

            recall = len(gold_urls & retrieved_urls) / len(gold_urls)
            recalls.append(recall)

        results[name] = sum(recalls) / len(recalls)

    return results

ablation_results = evaluate_retrievers(eval_dataset)
print(ablation_results)


{'dense': 0.9, 'sparse': 0.9, 'hybrid': 0.9}


## Error Analysis by Question Type

In [54]:
from collections import defaultdict

def error_analysis_by_question_type(eval_dataset, top_k=5):

    results = defaultdict(lambda: {
        "count": 0,
        "recall_sum": 0,
        "faithful_sum": 0
    })

    for item in eval_dataset:

        question = item["question"]
        qtype = item["question_type"]
        gold_urls = set(item["source_urls"])

        # Retrieve context
        retrieved_chunks = hybrid_retrieve(question, top_k=top_k)
        retrieved_urls = set(chunk["url"] for chunk in retrieved_chunks)

        # Compute Recall
        recall = len(gold_urls & retrieved_urls) / len(gold_urls)

        # Prepare context for faithfulness
        MAX_CONTEXT_CHARS = 1200   # safe for TinyLlama

        context = ""

        for chunk in retrieved_chunks:

            text = chunk["text"]

            if len(context) + len(text) > MAX_CONTEXT_CHARS:
                break

            context += text + "\n"


        # Generate answer
        generated_answer = generate_answer_from_context(question, context)

        # Faithfulness check
        faithful = faithfulness_check(llm, question, generated_answer, context)

        # Update stats
        results[qtype]["count"] += 1
        results[qtype]["recall_sum"] += recall
        results[qtype]["faithful_sum"] += faithful

    # Compute averages
    summary = {}

    for qtype, stats in results.items():
        summary[qtype] = {
            "num_questions": stats["count"],
            "avg_recall": stats["recall_sum"] / stats["count"],
            "avg_faithfulness": stats["faithful_sum"] / stats["count"]
        }

    return summary


## Error Analysis

In [55]:
error_summary = error_analysis_by_question_type(eval_dataset)

print("Error Analysis by Question Type:\n")

for qtype, stats in error_summary.items():

    print(f"Question Type: {qtype}")
    print(f"Number of Questions: {stats['num_questions']}")
    print(f"Average Recall: {stats['avg_recall']:.3f}")
    print(f"Average Faithfulness: {stats['avg_faithfulness']:.3f}")
    print("-" * 40)


Error Analysis by Question Type:

Question Type: factual
Number of Questions: 20
Average Recall: 0.900
Average Faithfulness: 0.750
----------------------------------------
Question Type: comparative
Number of Questions: 20
Average Recall: 0.850
Average Faithfulness: 0.700
----------------------------------------
Question Type: inferential
Number of Questions: 20
Average Recall: 0.950
Average Faithfulness: 0.650
----------------------------------------


In [56]:
import json

output_path = "/content/drive/MyDrive/Assignment2/error_analysis.json"

with open(output_path, "w") as f:
    json.dump(error_summary, f, indent=2)

print("Error analysis saved to:", output_path)


Error analysis saved to: /content/drive/MyDrive/Assignment2/error_analysis.json


## LLM-as-Judge evaluation function

In [57]:
def llm_judge_score(llm, question, generated_answer, ground_truth):

    prompt = f"""
You are an expert evaluator.

Question:
{question}

Ground Truth Answer:
{ground_truth}

Generated Answer:
{generated_answer}

Evaluate the generated answer on a scale from 0 to 1 based on:

- factual correctness
- completeness
- relevance

Return ONLY a number between 0 and 1.
"""

    response = llm(prompt)

    try:
        score = float(response.strip())
    except:
        score = 0.0

    return score


In [58]:
def compute_llm_judge_score(eval_dataset, top_k=5):

    scores = []

    for item in eval_dataset:

        question = item["question"]
        ground_truth = item["answer"]

        retrieved_chunks = hybrid_retrieve(question, top_k)
        #context = "\n".join(chunk["text"] for chunk in retrieved_chunks)
        MAX_CONTEXT_CHARS = 1500

        context = ""
        for chunk in retrieved_chunks:
            if len(context) + len(chunk["text"]) > MAX_CONTEXT_CHARS:
                break
            context += chunk["text"] + "\n"


        generated_answer = generate_answer_from_context(question, context)

        score = llm_judge_score(
            llm,
            question,
            generated_answer,
            ground_truth
        )

        scores.append(score)

    return sum(scores) / len(scores)


In [59]:
judge_score = compute_llm_judge_score(eval_dataset)

print("LLM-as-Judge Score:", judge_score)


LLM-as-Judge Score: 0.0


## Automated Pipeline

In [60]:
import time

def evaluate_single_question(item, top_k=5):

    question = item["question"]
    ground_truth = item["answer"]
    gold_urls = set(item["source_urls"])

    start = time.time()

    # Retrieval
    retrieved_chunks = hybrid_retrieve(question, top_k)

    #context = "\n".join(chunk["text"] for chunk in retrieved_chunks)
    MAX_CONTEXT_CHARS = 1500

    context = ""
    for chunk in retrieved_chunks:
        text = chunk["text"]
        if len(context) + len(chunk["text"]) > MAX_CONTEXT_CHARS:
           break
        context += chunk["text"] + "\n"

    retrieved_urls = set(chunk["url"] for chunk in retrieved_chunks)

    # Answer generation
    generated_answer = generate_answer_from_context(question, context)

    # Recall
    recall = len(gold_urls & retrieved_urls) / len(gold_urls)

    # Faithfulness
    faithfulness = faithfulness_check(
        llm,
        question,
        generated_answer,
        context
    )

    # LLM judge
    judge_score = llm_judge_score(
        llm,
        question,
        generated_answer,
        ground_truth
    )

    end = time.time()

    return {
        "question": question,
        "ground_truth": ground_truth,
        "generated_answer": generated_answer,
        "recall": recall,
        "faithfulness": faithfulness,
        "judge_score": judge_score,
        "response_time": end - start
    }


In [61]:
import json
import pandas as pd

def run_full_evaluation(eval_dataset):

    results = []

    for item in eval_dataset:

        result = evaluate_single_question(item)

        results.append(result)

    df = pd.DataFrame(results)

    summary = {

        "avg_recall": df["recall"].mean(),
        "avg_faithfulness": df["faithfulness"].mean(),
        "avg_judge_score": df["judge_score"].mean(),
        "avg_response_time": df["response_time"].mean()

    }

    return df, summary


In [62]:
results_df, summary = run_full_evaluation(eval_dataset)

print(summary)


{'avg_recall': np.float64(0.9), 'avg_faithfulness': np.float64(0.6666666666666666), 'avg_judge_score': np.float64(0.0), 'avg_response_time': np.float64(3.9614266753196716)}


### Saving the evaluation details

In [63]:
results_df.to_csv(
    "/content/drive/MyDrive/Assignment2/evaluation_results.csv",
    index=False
)

with open(
    "/content/drive/MyDrive/Assignment2/evaluation_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)


### Single Question Evaluation

In [64]:
import json
import pandas as pd
import time

# =========================
# Single Question Evaluation
# =========================

def evaluate_single_question(item, top_k=5):

    question = item["question"]
    ground_truth = item["answer"]
    gold_urls = set(item["source_urls"])
    qtype = item.get("question_type", "unknown")

    start = time.time()

    # Hybrid retrieval
    retrieved_chunks = hybrid_retrieve(question, top_k)

    #context = "\n".join(chunk["text"] for chunk in retrieved_chunks)
    MAX_CONTEXT_CHARS = 1500

    context = ""
    for chunk in retrieved_chunks:
        text = chunk["text"]
        if len(context) + len(chunk["text"]) > MAX_CONTEXT_CHARS:
           break
        context += chunk["text"] + "\n"

    retrieved_urls = []
    seen = set()
    for chunk in retrieved_chunks:
        url = chunk["url"]
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)

    # Generate answer
    generated_answer = generate_answer_from_context(question, context)

    # =====================
    # Recall@K
    # =====================
    recall = len(set(retrieved_urls) & gold_urls) / len(gold_urls)

    # =====================
    # MRR (URL-level)
    # =====================
    mrr = 0
    for rank, url in enumerate(retrieved_urls, start=1):
        if url in gold_urls:
            mrr = 1 / rank
            break

    # =====================
    # Faithfulness
    # =====================
    faithfulness = faithfulness_check(
        llm,
        question,
        generated_answer,
        context
    )

    # =====================
    # LLM Judge Score
    # =====================
    judge_score = llm_judge_score(
        llm,
        question,
        generated_answer,
        ground_truth
    )

    end = time.time()

    return {

        "question": question,
        "question_type": qtype,
        "ground_truth": ground_truth,
        "generated_answer": generated_answer,

        "recall": recall,
        "mrr": mrr,
        "faithfulness": faithfulness,
        "judge_score": judge_score,

        "response_time": end - start

    }


# =========================
# Full Evaluation Pipeline
# =========================

def run_complete_pipeline(eval_dataset):

    print("Starting full evaluation pipeline...\n")

    results = []

    # Evaluate each question
    for item in eval_dataset:

        result = evaluate_single_question(item)

        results.append(result)

    results_df = pd.DataFrame(results)

    # =====================
    # Summary Metrics
    # =====================

    summary = {

        "avg_recall": results_df["recall"].mean(),
        "avg_mrr": results_df["mrr"].mean(),
        "avg_faithfulness": results_df["faithfulness"].mean(),
        "avg_judge_score": results_df["judge_score"].mean(),
        "avg_response_time": results_df["response_time"].mean()

    }

    # =====================
    # Ablation Study
    # =====================

    print("Running ablation study...")

    ablation_results = evaluate_retrievers(eval_dataset)

    # =====================
    # Error Analysis
    # =====================

    print("Running error analysis...")

    error_results = error_analysis_by_question_type(eval_dataset)

    # =====================
    # Save Structured Outputs
    # =====================

    base_path = "/content/drive/MyDrive/Assignment2/"

    results_df.to_csv(
        base_path + "evaluation_results.csv",
        index=False
    )

    with open(base_path + "evaluation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    with open(base_path + "ablation_results.json", "w") as f:
        json.dump(ablation_results, f, indent=2)

    with open(base_path + "error_analysis.json", "w") as f:
        json.dump(error_results, f, indent=2)

    print("\nPipeline completed successfully.")

    print("\nSummary Metrics:")
    for k, v in summary.items():
        print(f"{k}: {v:.4f}")

    return results_df, summary, ablation_results, error_results



In [65]:
# Run evaluation pipeline on small subset to avoid rate limit

subset = eval_dataset[:10]

results_df, summary, ablation_results, error_results = run_complete_pipeline(subset)

print("\nFinal Summary:")
print(summary)


Starting full evaluation pipeline...

Running ablation study...
Running error analysis...

Pipeline completed successfully.

Summary Metrics:
avg_recall: 0.7000
avg_mrr: 0.7000
avg_faithfulness: 0.6000
avg_judge_score: 0.0000
avg_response_time: 4.0438

Final Summary:
{'avg_recall': np.float64(0.7), 'avg_mrr': np.float64(0.7), 'avg_faithfulness': np.float64(0.6), 'avg_judge_score': np.float64(0.0), 'avg_response_time': np.float64(4.043768358230591)}
